# 04 — the kriging-multigrid demo implementation, ported to the bed

Notebook 03 built the bed ensemble with the **published `multigrid_sgsim` package**
(multigrid + sequential Gaussian simulation). This notebook ports the **other**
implementation of the method — the one in
`research/mx/multigrid_demo/06102024_AutoMultigridSim_demo.ipynb`, where each
realization is built with **kriging** (the WestPilbara production notebook is a
different animal — SGeMS sgsim per level; see the addendum at the end) —
and runs it on the same conditioning data (the gridded radar bed picks from
notebook 02, datum-aligned exactly as in notebook 03).

Contents:

1. a faithful port of the demo primitives (stratified-RBF trend draw,
   `multigrid_smpl`, per-level simple kriging with the `min(spacing, fitted range)`
   rule), traced level by level;
2. an N=10 ensemble from it, compared head-to-head with the notebook-03 SGS
   ensemble (galleries, metrics, cross-section, spread vs distance);
3. a written account of **what actually differs** between the package and the
   demo implementations, and why the demo's maps look so much nicer;
4. a third generator — the **conditional-GRF hybrid** (the demo notebook's own,
   unused `conditionalGRF_pts2grd` idea) on the published-bed trend — which turns
   out to combine the demo's smoothness with the package's honest statistics.

Runs in a Python env with `gstools`, `skgstat` and `scipy` installed.

## Package vs demo: what is actually different

Both are "MGSIM": a coarse→fine cascade over subsets of the conditioning data,
with the residual recomputed against the running field between levels, and a
variogram setting the correlation range at each level. Everything else moved:

| | `multigrid_sgsim` package (nb 03) | AutoMultigridSim demo (this one) |
|---|---|---|
| residual simulator per level | **sequential Gaussian simulation** (gstatsim `okrige_sgs`), in normal-score space | **deterministic simple kriging** (gstools `krige.Simple`, exact) |
| where the randomness lives | the SGS path — every node is a conditional *draw* | the **trend draw** (stratified subsample → thin-plate RBF) + the random lattice offsets in `multigrid_smpl` |
| normal-score transform | yes (QuantileTransformer, clipped) | no — raw units throughout |
| per-level range | one global fitted variogram | re-fitted each level, then **`min(level spacing, fitted range)`** |
| texture of one realization | variogram-consistent grain everywhere (visually noisy) | analytically smooth (Gaussian-covariance kriging surfaces) |
| statistical character | a draw from the conditional distribution: departure-from-trend variance ≈ data variance, spread → sill away from data | an ensemble of *smoothers*: each member under-disperses; ensemble spread is whatever the trend draw + offsets inject |

Two non-obvious consequences, both demonstrated below on the radar-line data:

- **The demo's range rule doubles as a numerical stabilizer.** Capping the range
  at the level spacing keeps every kriging matrix well-conditioned (neighbors at
  distance ≈ range are only ~0.46 correlated). Krige the same line data with the
  full fitted range and no nugget and the system is near-singular — the weights
  explode (we measured mid-gap excursions of ±300 m before adding the 1% nugget
  that notebook 03 also needed for exactly this reason).
- **The demo datasets hide the failure modes.** The demo's and the WestPilbara
  survey's lines sit ~4 px apart on their grids (400 m lines, 100 m cells) — gaps
  far inside the variogram range, with residuals still spatially correlated at the
  finest level. Our radar
  lines sit 5–40+ px apart with a fine-scale residual that fits as ~white noise
  (effective range 0.3 px). Exact kriging of that noise imprints bull's-eye chains
  along the flight tracks, and the RBF trend draw wanders freely in the gaps —
  where notebook 03 instead keeps the published bed's information.

In [ ]:
import json, time
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.interpolate import RBFInterpolator
from scipy.spatial import cKDTree
from scipy.spatial.distance import cdist
from scipy.linalg import cho_factor, cho_solve
import skgstat as skg
import gstools as gst
import matplotlib.pyplot as plt

N_GRID, PX_M = 166, 150.0
SRC, DERIVED, FIGS = Path('../data/source_copies'), Path('../data/derived'), Path('../figures')

def read_dat(p):
    return np.loadtxt(p)[:, 2].reshape(N_GRID, N_GRID)

trend_pub = read_dat(SRC / '2019cw428_bed_d25km_h1004_a0.792.dat')
surface   = read_dat(SRC / '2019cw428_surface_d25km_h1004_a0.792.dat')

# conditioning data: identical preparation to notebook 03
gz = np.load(DERIVED / 'gridded_bed_cw1951_150m.npz')
I0, I1 = json.loads(str(gz['meta']))['dem_sub_lattice']
radar = gz['bed_hm'][I0:I1 + 1, I0:I1 + 1]
shift = float(np.nanmedian(radar - trend_pub))          # ellipsoid->published datum
resid0 = radar - shift - trend_pub
lo, hi = np.nanpercentile(resid0, [0.25, 99.75])        # same outlier trim as nb 03
m = np.isfinite(resid0) & (resid0 >= lo) & (resid0 <= hi)
iy, ix = np.where(m)
vals = (radar - shift)[m]                               # aligned bed values
r_obs = resid0[m]                                       # residuals vs published bed
xyv = np.column_stack([ix, iy, vals]).astype(float)
P = xyv[:, :2]

# the notebook-03 ensemble, for the head-to-head
R3 = np.load(DERIVED / 'bed_realizations_cw1951.npz')
sgs_beds, taper_w = R3['beds'], R3['taper']

print(f'conditioning: {len(vals)} nodes · datum shift {shift:+.1f} m · '
      f'resid-vs-published std {r_obs.std():.1f} m')
print(f'nb-03 ensemble loaded: beds {sgs_beds.shape}, keys {R3.files}')

## A note on how the kriging is computed

The demo calls `gstools.krige.Simple(..., exact=True)` directly. gstools' exact
path computes `K⁻¹·k(x)` **per target node** (O(n²) per node in its cython
kernel): fine for the demo's few-hundred-point levels, but at our finest level
(5,177 conditioning nodes × 27,556 targets ≈ 7×10¹⁴ flops) a profiled run was on
track for **hours**. The estimate itself doesn't need any of that: simple kriging
is `ẑ(x) = k(x)ᵀ·w` with `w = K⁻¹c` computed **once**. `krige_grid` below
factorizes the covariance matrix a single time (Cholesky, with a jitter ladder
for near-singular systems) and evaluates the grid in seconds. The estimate is
identical to gstools' — only the algebra is reordered.

In [ ]:
# --- the demo primitives, ported ---------------------------------------------

def rbf_trend(xyv, rng, block=10, smoothing=1.0):
    """Demo trend *draw*: one random pick per coarse block (their stratified
    sample), thin-plate RBF through the picks. Randomized per realization."""
    strata = (xyv[:, 0] // block).astype(int) * 1000 + (xyv[:, 1] // block).astype(int)
    pick = [rng.choice(np.where(strata == s)[0]) for s in np.unique(strata)]
    p = xyv[np.array(pick)]
    gy, gx = np.mgrid[0:N_GRID, 0:N_GRID]
    tgt = np.column_stack([gx.ravel(), gy.ravel()]).astype(float)
    f = RBFInterpolator(p[:, :2], p[:, 2], smoothing=smoothing, kernel='thin_plate_spline')
    return f(tgt).reshape(N_GRID, N_GRID), len(p)

def multigrid_smpl(xyv, spacing, rng):
    """Demo sampler: nearest data point to each node of a randomly offset lattice
    at `spacing`; retried until the subsample keeps enough variance (degrading
    threshold), so low-variance subsets get rejected."""
    var_data = xyv[:, 2].var()
    coeff, tries = 1.1, 0
    tree = cKDTree(xyv[:, :2])
    while True:
        x0 = rng.integers(0, spacing + 1); y0 = rng.integers(0, spacing + 1)
        cx, cy = np.meshgrid(np.arange(x0, N_GRID, spacing), np.arange(y0, N_GRID, spacing))
        _, idx = tree.query(np.column_stack([cx.ravel(), cy.ravel()]))
        sub = xyv[np.unique(idx)]
        tries += 1
        if sub[:, 2].var() >= var_data * coeff or tries > spacing * 20:
            if tries > spacing * 20:
                coeff = max(0.7, coeff - 0.1)
            if sub[:, 2].var() >= var_data * coeff or tries > spacing * 60:
                return sub

def krige_grid(sub, var, r):
    """Simple kriging (mean 0) of sub[:,2] onto the full grid — identical
    estimate to gstools krige.Simple(exact=True), via one-time Cholesky."""
    model = gst.Gaussian(dim=2, var=var, len_scale=r)
    Q = sub[:, :2]
    C = model.covariance(cdist(Q, Q))
    cf = None
    for jit in (1e-8, 1e-6, 1e-4, 1e-2):
        try:
            cf = cho_factor(C + jit * var * np.eye(len(Q)), lower=True)
            break
        except np.linalg.LinAlgError:
            continue
    w = cho_solve(cf, sub[:, 2])
    gy, gx = np.mgrid[0:N_GRID, 0:N_GRID]
    tgt = np.column_stack([gx.ravel(), gy.ravel()]).astype(float)
    out = np.empty(len(tgt))
    for a in range(0, len(tgt), 8000):
        out[a:a + 8000] = model.covariance(cdist(tgt[a:a + 8000], Q)) @ w
    return out.reshape(N_GRID, N_GRID), jit

def var_range(sub, spacing, rng):
    """Demo range rule: refit a Gaussian variogram on this level's residual
    subsample, then use min(level spacing, fitted effective range)."""
    s = sub if len(sub) <= 3000 else sub[rng.choice(len(sub), 3000, replace=False)]
    V = skg.Variogram(s[:, :2], s[:, 2], model='gaussian', maxlag='median',
                      n_lags=20, normalize=False)
    r = float(V.describe()['effective_range'])
    return r, min(r, float(spacing))

MG_SPACINGS = [12, 6, 3, 1]   # px; 150 m px -> 1.8 km down to node scale

def kmg_realization(seed, verbose=False):
    rng = np.random.default_rng(seed)
    tr, npts = rbf_trend(xyv, rng)
    field = tr.copy()
    stages = [('trend draw (%d pts)' % npts, tr.copy())]
    for spacing in MG_SPACINGS:
        res = xyv.copy()
        res[:, 2] = vals - field[iy, ix]
        sub = res if spacing == 1 else multigrid_smpl(res, spacing, rng)
        r_fit, r_use = var_range(sub, spacing, rng)
        kf, jit = krige_grid(sub, res[:, 2].var(), max(r_use, 2.0))
        field = field + kf
        if verbose:
            err = np.abs(field[iy, ix] - vals)
            print(f'  level {spacing:2d}px: {len(sub):5d} pts · fitted range {r_fit:5.1f}px'
                  f' -> used {max(r_use, 2.0):4.1f}px · jitter {jit:g} · '
                  f'cond-err median {np.median(err):5.2f} m')
        stages.append((f'+ level {spacing} px', field.copy()))
    return field, stages

In [ ]:
# --- one realization, traced level by level ----------------------------------
t0 = time.time()
print('seed 0 cascade:')
field0, stages = kmg_realization(0, verbose=True)
print(f'[{time.time()-t0:.0f} s]')

vmin, vmax = np.percentile(trend_pub, [2, 98])
fig, axs = plt.subplots(1, 5, figsize=(21, 4.4), constrained_layout=True)
for ax, (t, b) in zip(axs, stages):
    im = ax.imshow(b, origin='lower', cmap='terrain', vmin=vmin, vmax=vmax)
    ax.set_title(t, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
fig.colorbar(im, ax=axs, shrink=0.8, label='bed z (m)')
fig.suptitle('demo kriging-MG cascade, seed 0: trend draw, then residual kriging at 1.8 km / 900 m / 450 m / 150 m', fontsize=11)
fig.savefig(FIGS / 'kmg_cascade.png', dpi=110, bbox_inches='tight')
plt.show()

In [ ]:
# --- N=10 ensemble from the port; same 2 km cosine wall taper as nb 03 -------
N_REAL = 10

def apply_taper(bed):
    """Blend toward the *published* bed at the walls (identical taper weights to
    nb 03) so wall BCs would stay fixed across any ensemble."""
    return trend_pub + taper_w * (bed - trend_pub)

t0 = time.time()
kmg_beds = []
for i in range(N_REAL):
    f, _ = kmg_realization(i)
    kmg_beds.append(apply_taper(f))
    print(f'seed {i}: done [{time.time()-t0:.0f} s cumulative]')
kmg_beds = np.array(kmg_beds)

In [ ]:
# --- head-to-head metrics + gallery vs the nb-03 SGS ensemble ----------------
core = slice(20, -20)                       # clear of the taper zone (~13 px)
in_core = (ix >= 20) & (ix < N_GRID - 20) & (iy >= 20) & (iy < N_GRID - 20)

def rough(b):
    d2 = np.diff(b, n=2, axis=1)[core, core]
    return float(np.sqrt(np.mean(d2 ** 2)))

def metrics(beds, name):
    err = np.abs(beds[:, iy[in_core], ix[in_core]] - vals[in_core])
    return dict(method=name,
                rough_m=np.mean([rough(b) for b in beds]),
                dep_std_m=np.mean([np.std((b - trend_pub)[core, core]) for b in beds]),
                cond_err_med_m=float(np.median(err)),
                cond_err_p95_m=float(np.percentile(err, 95)),
                ens_std_core_m=float(beds.std(0)[core, core].mean()))

rows = [metrics(sgs_beds, 'MGSIM package (nb 03)'),
        metrics(kmg_beds, 'demo kriging-MG (port)')]
print(f'reference: published-bed roughness {rough(trend_pub):.1f} m · '
      f'radar-resid std {r_obs.std():.1f} m\n')
print(pd.DataFrame(rows).round(1).to_string(index=False))

fig, axs = plt.subplots(2, 5, figsize=(21, 8.6), constrained_layout=True)
for row, (beds, label) in enumerate([(sgs_beds, 'MGSIM package (nb 03)'),
                                     (kmg_beds, 'demo kriging-MG (port)')]):
    for j in range(4):
        im = axs[row, j].imshow(beds[j], origin='lower', cmap='terrain', vmin=vmin, vmax=vmax)
        axs[row, j].set_title(f'{label} · seed {j}', fontsize=9)
    ims = axs[row, 4].imshow(beds.std(0), origin='lower', cmap='magma', vmin=0, vmax=80)
    axs[row, 4].set_title(f'{label} · ensemble std', fontsize=9)
    for ax in axs[row]:
        ax.set_xticks([]); ax.set_yticks([])
fig.colorbar(im, ax=axs[:, :4], shrink=0.7, label='bed z (m)')
fig.colorbar(ims, ax=axs[:, 4], shrink=0.7, label='std (m)')
fig.savefig(FIGS / 'kmg_gallery.png', dpi=110, bbox_inches='tight')
plt.show()

## Reading the head-to-head

- The port's members condition **exactly** (cond-err 0): exactness at the cell
  values is not a virtue here — the finest level's residual fits as ~white noise,
  so exact kriging of it stamps 2-px bull's-eyes along every flight track (visible
  as chains in the gallery, and as the elevated roughness metric).
- Its departure-from-published std runs ~2× the data's own residual std: that is
  the RBF **trend draw** wandering in the multi-km data gaps. The demo *intends*
  the trend draw to carry the ensemble spread — on dense, regular sampling it
  wanders by about the right amount; on line data with an informative published
  bed available, it both overshoots and throws away real information.
- The SGS ensemble keeps the published bed as its trend, so mid-gap members stay
  anchored (departure std ≈ data std) — but every member wears variogram grain.
- Ensemble spread: the port's mid-gap spread is trend-draw spread (large,
  weakly controlled); the SGS spread rises to ~the sill and is
  variogram-controlled. Cross-section and spread-vs-distance below.

## The third generator: conditional GRF on the published trend

The demo notebook defines (but never uses in its loop) `conditionalGRF_pts2grd` —
gstools' conditioned random field: kriging **plus** a Gaussian random field
carrying the kriging error. That is the textbook conditional simulation, and it is
exactly the middle path for this dataset:

- trend = the **published bed** (keep its information, as in nb 03);
- residuals conditioned by simple kriging with the fitted Gaussian variogram;
- each realization adds an unconditional GRF minus its own kriged interpolation,
  so members pass through the data (to within the nugget), fluctuate with the
  full sill mid-gap, and are as smooth as the covariance — no SGS grain, no
  track-line stamps.

Computed as `bed = trend + k(r_obs) + [g − k(g at data)]` with one shared
Cholesky factorization (seconds per member). The raw-space fit returns **zero
nugget**, which leaves the dense-line covariance matrix near-singular — kriging
weights explode (±300 m mid-gap in a pilot). We therefore reuse notebook 03's 1%
relative nugget; the conditioning error it induces (median ≈ 2σ_nugget) is the
same size as the SGS ensemble's.

In [ ]:
# --- conditional-GRF ensemble ------------------------------------------------
rng = np.random.default_rng(0)
s = rng.choice(len(r_obs), 3000, replace=False)
V = skg.Variogram(P[s], r_obs[s], model='gaussian', maxlag='median', n_lags=20, normalize=False)
desc = V.describe()
r_eff, sill = float(desc['effective_range']), float(desc['sill'])
nug_fit = float(desc.get('nugget', 0.0))
jit = max(nug_fit, 0.01 * sill)
print(f'raw-residual gaussian fit: eff range {r_eff:.1f} px ({r_eff*PX_M/1000:.1f} km) · '
      f'sill {sill:.0f} m² (σ {np.sqrt(sill):.1f} m) · fitted nugget {nug_fit:.0f}')
print(f'regularization: 1% relative nugget -> σ_nugget {np.sqrt(jit):.1f} m')

# gstools Gaussian: cor(h)=exp(-(pi/4)(h/l)^2) -> 95% decorrelation at 1.954*l
model = gst.Gaussian(dim=2, var=sill, len_scale=r_eff / 1.954)
t0 = time.time()
C = model.covariance(cdist(P, P))
cf = cho_factor(C + jit * np.eye(len(P)), lower=True)
gy, gx = np.mgrid[0:N_GRID, 0:N_GRID]
tgt = np.column_stack([gx.ravel(), gy.ravel()]).astype(float)
Kt = np.empty((len(tgt), len(P)))
for a in range(0, len(tgt), 8000):
    Kt[a:a + 8000] = model.covariance(cdist(tgt[a:a + 8000], P))

def krige(vals_at_P):
    return (Kt @ cho_solve(cf, vals_at_P)).reshape(N_GRID, N_GRID)

k_data = krige(r_obs)
print(f'factorization + kriged data residual [{time.time()-t0:.0f} s]')

grf_beds = []
for seed in range(N_REAL):
    srf = gst.SRF(model, seed=seed)
    g = srf((np.arange(N_GRID), np.arange(N_GRID)), mesh_type='structured').T
    bed = trend_pub + k_data + (g - krige(g[iy, ix]))
    grf_beds.append(apply_taper(bed))
grf_beds = np.array(grf_beds)
print(f'{N_REAL} conditional-GRF realizations [{time.time()-t0:.0f} s]')

rows.append(metrics(grf_beds, 'conditional GRF (hybrid)'))
print()
print(pd.DataFrame(rows).round(1).to_string(index=False))

fig, axs = plt.subplots(1, 5, figsize=(21, 4.4), constrained_layout=True)
for j in range(4):
    im = axs[j].imshow(grf_beds[j], origin='lower', cmap='terrain', vmin=vmin, vmax=vmax)
    axs[j].set_title(f'conditional GRF · seed {j}', fontsize=9)
ims = axs[4].imshow(grf_beds.std(0), origin='lower', cmap='magma', vmin=0, vmax=80)
axs[4].set_title('conditional GRF · ensemble std', fontsize=9)
for ax in axs:
    ax.set_xticks([]); ax.set_yticks([])
fig.colorbar(im, ax=axs[:4], shrink=0.8, label='bed z (m)')
fig.colorbar(ims, ax=axs[4], shrink=0.8, label='std (m)')
fig.savefig(FIGS / 'condgrf_gallery.png', dpi=110, bbox_inches='tight')
plt.show()

In [ ]:
# --- spread vs distance-to-data, and a flowline cross-section ----------------
ENS = [(sgs_beds, 'MGSIM package (nb 03)', 'tab:orange'),
       (kmg_beds, 'demo kriging-MG (port)', 'tab:red'),
       (grf_beds, 'conditional GRF (hybrid)', 'tab:blue')]

tree = cKDTree(P)
dmap = tree.query(tgt)[0].reshape(N_GRID, N_GRID)
bins = np.arange(0, 42, 2)
fig, ax = plt.subplots(figsize=(8, 5))
for beds, name, c in ENS:
    sd = beds.std(0)
    bc, prof = [], []
    for b0, b1 in zip(bins[:-1], bins[1:]):
        sel = (dmap[core, core] >= b0) & (dmap[core, core] < b1)
        if sel.sum() > 30:
            bc.append((b0 + b1) / 2 * PX_M / 1000)
            prof.append(sd[core, core][sel].mean())
    ax.plot(bc, prof, '-o', ms=3, color=c, label=name)
ax.axhline(np.sqrt(sill), ls=':', c='gray', label=f'radar-resid σ ({np.sqrt(sill):.0f} m)')
ax.set_xlabel('distance to nearest radar cell (km)')
ax.set_ylabel('ensemble std (m)')
ax.set_title('where each ensemble puts its uncertainty')
ax.legend(fontsize=9)
fig.savefig(FIGS / 'kmg_spread_vs_distance.png', dpi=110, bbox_inches='tight')
plt.show()

IY = 82   # same flowline row as nb 03
x_km = np.arange(N_GRID) * PX_M / 1000
fig, axs = plt.subplots(3, 1, figsize=(12, 10), sharex=True, sharey=True, constrained_layout=True)
for ax, (beds, name, c) in zip(axs, ENS):
    ax.fill_between(x_km, beds[:, IY, :].min(0), beds[:, IY, :].max(0), alpha=.3, color=c)
    ax.plot(x_km, beds[:, IY, :].mean(0), color=c, lw=1.2, label=f'{name}: mean, min–max of {N_REAL}')
    ax.plot(x_km, trend_pub[IY], 'k-', lw=1.2, label='published bed')
    row = iy == IY
    ax.plot(ix[row] * PX_M / 1000, vals[row], 'k.', ms=6, label='radar (gridded, aligned)')
    ax.legend(loc='upper right', fontsize=8)
    ax.set_ylabel('bed z (m)')
axs[-1].set_xlabel('along-flow x (km)')
axs[0].set_title(f'flowline cross-section, row iy={IY}')
fig.savefig(FIGS / 'kmg_xsection.png', dpi=110, bbox_inches='tight')
plt.show()

In [ ]:
# --- one-figure summary ------------------------------------------------------
fig, axs = plt.subplots(1, 4, figsize=(19, 4.6), constrained_layout=True)
panels = [(trend_pub, 'published bed'), (sgs_beds[0], 'MGSIM package (nb 03)'),
          (kmg_beds[0], 'demo kriging-MG (port)'), (grf_beds[0], 'conditional GRF (hybrid)')]
for ax, (b, t) in zip(axs, panels):
    im = ax.imshow(b, origin='lower', cmap='terrain', vmin=vmin, vmax=vmax)
    ax.set_title(t, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
fig.colorbar(im, ax=axs, shrink=0.8, label='bed z (m)')
fig.savefig(FIGS / 'methods_compare.png', dpi=110, bbox_inches='tight')
plt.show()

## Verdict, and what feeds the Elmer ensemble

- **Demo kriging-MG port**: beautiful *between* the lines, but on this geometry
  it stamps flight-track bull's-eyes and lets the trend draw invent ±100 m
  features in the gaps where the published bed already knows better. Not suitable
  as an Elmer bed — the stress ensemble would inherit artifacts pinned to the
  radar acquisition pattern. (Saved to figures only; no DEMs written.)
- **Package SGS (nb 03)**: statistically honest and artifact-free; the grain is
  variogram-consistent texture, not error. This is the published MGSIM method —
  the defense-website story — and remains the default DEM set
  (`dems/*_mgsim??.dat`).
- **Conditional GRF (hybrid)**: as smooth as the demo's maps *and* as honest as
  the package's statistics, using a primitive that already lives in the demo
  notebook. Saved as an alternate DEM set (`dems/*_condgrf??.dat`) — switching
  the Elmer ensemble to it is a filename change.

For the *surface-stress* spread itself the SGS-vs-GRF choice should matter
little: membrane stresses in ~1 km-thick ice filter bed wavelengths ≲ one ice
thickness, which is exactly where the two generators differ (node-scale grain
vs smoothness). The km-scale structure — which both constrain identically near
the lines and spread mid-gap — is what will drive the stress ensemble.

In [ ]:
# --- save the conditional-GRF set: npz + sif-format DEMs (gitignored) --------
MIN_ICE = 20.0
clips = int((grf_beds > surface - MIN_ICE).sum())
grf_beds = np.minimum(grf_beds, surface - MIN_ICE)
print(f'min-ice guard ({MIN_ICE:.0f} m): {clips} node-clips across the ensemble')

meta = dict(method='conditional GRF on published-bed trend (demo conditionalGRF_pts2grd concept)',
            trend='published bed DEM (source_copies)', datum_shift_m=shift,
            trim_pct=[0.25, 99.75], vario=dict(model='gaussian', eff_range_px=r_eff,
            sill_m2=sill, nugget_rel=0.01), seeds=list(range(N_REAL)),
            taper='nb-03 cosine 2 km to published bed', min_ice_m=MIN_ICE)
np.savez(DERIVED / 'bed_realizations_cw1951_condgrf.npz',
         beds=grf_beds, trend=trend_pub, surface=surface, taper=taper_w,
         obs_mask=m, meta=json.dumps(meta))

xy = np.loadtxt(SRC / '2019cw428_bed_d25km_h1004_a0.792.dat')[:, :2]
dem_dir = DERIVED / 'dems'
dem_dir.mkdir(exist_ok=True)
for i, bed in enumerate(grf_beds):
    np.savetxt(dem_dir / f'2019cw428_bed_d25km_h1004_a0.792_condgrf{i:02d}.dat',
               np.column_stack([xy, bed.ravel()]), fmt='%f %f %f')
print(f'saved bed_realizations_cw1951_condgrf.npz + {N_REAL} DEMs to {dem_dir}/')

## Addendum — what the WestPilbara production notebook actually does

Checked against `WestPilbara_deg135_Grid100m_400mLines_rotate.ipynb` and its
`sgems_par` / `sgems_sgsim_res` files once the data landed. It is **not** the
kriging cascade above. Each multigrid level's residual is simulated with
**SGeMS `sgsim`** — Simple Kriging, 12 max conditioning data, search ellipsoid
≈ the whole grid, normal-score transform via the hard-data `nonParamCdf` —
with a Gaussian variogram, **zero nugget**, and a range re-set per level:
42 / 6 / 2 cells for the 400 / 200 / 100 m levels on the rotated 100 m grid
(≈10× / 3× / 2× the level spacing). Ensemble member *i* = stratified-RBF trend
draw *i* + sgsim realization *i* at each level, residuals recomputed per member
between levels; the headline map is the E-type **mean** of 20 members.

Two details matter for reading its maps:

- the sgsim seed (`14071789`) is identical for every member, so the sequential
  random path and Gaussian deviates are shared across the ensemble — the
  stochastic component is perfectly correlated between members, and
  member-to-member spread is carried almost entirely by the trend draw;
- with full 100 m along-line posting and lines 4 cells apart, the SK variance at
  most nodes is tiny, so each sgsim pass is interpolation-dominated; that, plus
  hillshaded rendering of a ~1,100 nT signal carrying ±100 nT residuals, is
  where the smooth look comes from — not from the simulator.

Methodologically WestPilbara sits *between* the package (SGS per level, but one
global variogram and a fresh seed per realization) and this notebook's kriging
demo (per-level variogram refit, trend-draw ensemble).